# FonRadar - Data Collection and Exploration

In this notebook, we will:
1. Fetch fund data from TEFAS
2. Explore the dataset
3. Create initial visualizations
4. Calculate basic performance metrics

## 1. Import Libraries

In [ ]:
# Required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")

## 2. Fetch Data from TEFAS

In [ ]:
# Import our custom module
import sys
sys.path.append('..')

from src.tefas_client import TEFASClient

# Create client
client = TEFASClient()

# Display available funds
print("Funds to fetch:")
print("-" * 60)
for code, name in client.POPULAR_FUNDS.items():
    print(f"{code}: {name}")

In [ ]:
# Fetch data (this may take 1-2 minutes)
df = client.get_multiple_funds()

# Save data
df.to_csv('../data/fund_data.csv', index=False)
print("\n💾 Data saved to: data/fund_data.csv")

## 3. Data Exploration

In [ ]:
# If data already exists, you can load from file:
# df = pd.read_csv('../data/fund_data.csv', parse_dates=['date'])

# First 5 rows
df.head()

In [ ]:
# Dataset info
print("Dataset Shape:", df.shape)
print("\nColumn Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Summary statistics per fund
summary = df.groupby('fund_code').agg({
    'date': ['min', 'max', 'count'],
    'price': ['min', 'max', 'mean'],
    'category_rank': 'last'
}).round(2)

summary.columns = ['Start Date', 'End Date', 'Days', 
                   'Min Price', 'Max Price', 'Avg Price', 
                   'Category Rank']
summary

## 4. Visualizations

In [ ]:
# Price chart for all funds
fig, ax = plt.subplots(figsize=(14, 8))

for fund_code in df['fund_code'].unique():
    fund_data = df[df['fund_code'] == fund_code]
    ax.plot(fund_data['date'], fund_data['price'], label=fund_code, linewidth=1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Price (TRY)')
ax.set_title('Investment Fund Price History')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Normalized prices (for comparison)
# Set starting price to 100 for each fund

fig, ax = plt.subplots(figsize=(14, 8))

for fund_code in df['fund_code'].unique():
    fund_data = df[df['fund_code'] == fund_code].copy()
    fund_data = fund_data.sort_values('date')
    
    # Normalize to starting price
    first_price = fund_data['price'].iloc[0]
    fund_data['normalized_price'] = (fund_data['price'] / first_price) * 100
    
    ax.plot(fund_data['date'], fund_data['normalized_price'], label=fund_code, linewidth=1.5)

ax.axhline(y=100, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price (Start=100)')
ax.set_title('Comparative Fund Performance')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate daily returns
df = df.sort_values(['fund_code', 'date'])
df['daily_return'] = df.groupby('fund_code')['price'].pct_change() * 100

# Return distribution
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, fund_code in enumerate(df['fund_code'].unique()):
    fund_data = df[df['fund_code'] == fund_code]['daily_return'].dropna()
    
    axes[i].hist(fund_data, bins=50, edgecolor='black', alpha=0.7)
    axes[i].axvline(x=0, color='red', linestyle='--')
    axes[i].set_title(f'{fund_code}\nMean: {fund_data.mean():.2f}%, Std: {fund_data.std():.2f}%')
    axes[i].set_xlabel('Daily Return (%)')

plt.suptitle('Daily Return Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
# Correlation of daily returns between funds

# Create pivot table
pivot_df = df.pivot_table(
    index='date', 
    columns='fund_code', 
    values='daily_return'
)

# Calculate correlation
corr_matrix = pivot_df.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Fund Return Correlations')
plt.tight_layout()
plt.show()

## 5. Performance Metrics

In [ ]:
def calculate_metrics(df):
    """
    Calculate key performance metrics for each fund.
    
    Metrics:
    - Total Return: Overall percentage gain/loss
    - Annualized Return: Return adjusted to yearly basis
    - Volatility: Annualized standard deviation of returns
    - Sharpe Ratio: Risk-adjusted return measure
    - Maximum Drawdown: Largest peak-to-trough decline
    """
    metrics = []
    
    for fund_code in df['fund_code'].unique():
        fund = df[df['fund_code'] == fund_code].copy()
        fund = fund.sort_values('date')
        
        # Total return
        first_price = fund['price'].iloc[0]
        last_price = fund['price'].iloc[-1]
        total_return = ((last_price / first_price) - 1) * 100
        
        # Annualized return
        days = (fund['date'].iloc[-1] - fund['date'].iloc[0]).days
        annualized_return = total_return * (365 / days) if days > 0 else 0
        
        # Volatility (annualized)
        daily_std = fund['daily_return'].std()
        annualized_volatility = daily_std * np.sqrt(252)  # 252 trading days
        
        # Sharpe Ratio (risk-free rate ~40% for Turkey)
        risk_free_rate = 40
        sharpe = (annualized_return - risk_free_rate) / annualized_volatility if annualized_volatility > 0 else 0
        
        # Maximum Drawdown
        fund['cummax'] = fund['price'].cummax()
        fund['drawdown'] = (fund['price'] - fund['cummax']) / fund['cummax'] * 100
        max_drawdown = fund['drawdown'].min()
        
        metrics.append({
            'Fund Code': fund_code,
            'Total Return (%)': round(total_return, 2),
            'Annualized Return (%)': round(annualized_return, 2),
            'Volatility (%)': round(annualized_volatility, 2),
            'Sharpe Ratio': round(sharpe, 2),
            'Max Drawdown (%)': round(max_drawdown, 2)
        })
    
    return pd.DataFrame(metrics)

# Calculate metrics
metrics_df = calculate_metrics(df)
metrics_df = metrics_df.sort_values('Sharpe Ratio', ascending=False)
metrics_df

In [ ]:
# Risk-Return scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    metrics_df['Volatility (%)'],
    metrics_df['Annualized Return (%)'],
    s=100,
    c=metrics_df['Sharpe Ratio'],
    cmap='RdYlGn',
    edgecolors='black'
)

# Add fund labels
for i, row in metrics_df.iterrows():
    ax.annotate(
        row['Fund Code'],
        (row['Volatility (%)'], row['Annualized Return (%)']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

plt.colorbar(scatter, label='Sharpe Ratio')
ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Annualized Return (%)')
ax.set_title('Risk-Return Analysis')
ax.axhline(y=40, color='gray', linestyle='--', alpha=0.5, label='Risk-Free Rate (~40%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save metrics to CSV
metrics_df.to_csv('../data/fund_metrics.csv', index=False)
print("💾 Metrics saved to: data/fund_metrics.csv")

## 6. Key Findings

Based on our initial analysis:

1. **Performance Range**: [Describe the range of returns observed]

2. **Volatility Patterns**: [Note which fund types show higher/lower volatility]

3. **Correlations**: [Describe notable correlations between funds]

4. **Risk-Adjusted Returns**: [Note which funds have the best Sharpe Ratios]

*Fill in these observations after running the analysis with real data.*

## 7. Next Steps

- [ ] Add more funds to the analysis
- [ ] Create fund category classifications
- [ ] Compare against BIST-100 benchmark
- [ ] Add monthly/weekly aggregations
- [ ] Build machine learning models
- [ ] Create interactive dashboard